# 音频分析主笔记本

这是一个可重复使用、由配置驱动的本地音频/视频分析流程。请在中文输入配置单元中设置 `INPUT_MEDIA`，然后按顺序运行准备和分析单元。本地视频会通过 FFmpeg 转换为 WAV 分析音轨；流程不会下载媒体文件。生成的文件保存在 `results/master/<input-stem>/` 下。MERT 和 Basic Pitch 为可选步骤，需要安装相应依赖。

In [ ]:
!sudo apt-get install -y ffmpeg

In [ ]:
%pip install librosa scikit-learn demucs luigi

## 输入音频/视频文件（请先修改这里）

建议按账号 ID 整理媒体：放入 `video/<账号ID>/` 文件夹，并在下方填写账号 ID 和文件名。账号 ID 按文本记录（保留前导零）。每次运行分析一个文件；输出保存在 `results/master/<账号ID>/<文件名>/`，清单中也会记录账号 ID。项目级汇总清单保存在 `results/master/audio_manifest_all.csv`。若账号 ID 留空，则从 `video/` 根目录读取，结果沿用原来的目录结构。

In [ ]:
from pathlib import Path
import os

# ══════════════════════════════════════════════════════════════
#  ① 项目路径（自动识别，无需修改）
# ══════════════════════════════════════════════════════════════
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# ══════════════════════════════════════════════════════════════
#  ② 媒体根目录
#     默认是项目下的 video/ 文件夹。
#     也可在运行前用环境变量 MEDIA_SOURCE_DIR 指向其他位置。
# ══════════════════════════════════════════════════════════════
MEDIA_SOURCE_DIR = Path(os.environ.get("MEDIA_SOURCE_DIR", PROJECT_ROOT / "video")).expanduser().resolve()

# ══════════════════════════════════════════════════════════════
#  ③ 填写账号与文件名（★ 每次运行只需改这两行 ★）
#
#     目录约定：  <媒体根目录>/<账号ID>/<文件名>
#     输出位置：  results/master/<账号ID>/<文件名（不含扩展名）>/
#
#     例：视频放在  video/人民日报/xxx.mp4
#         则 ACCOUNT_ID = "人民日报"，INPUT_FILENAME = "xxx.mp4"
#
#     ACCOUNT_ID 留空 ("") 表示直接从媒体根目录读取文件，
#     结果保存在  results/master/<文件名>/。
#     也可用环境变量 ACCOUNT_ID / INPUT_FILENAME 覆盖（供批量运行使用）。
# ══════════════════════════════════════════════════════════════
ACCOUNT_ID = os.environ.get("ACCOUNT_ID", "")                      # ← 发布账号（如 "人民日报"）
INPUT_FILENAME = os.environ.get("INPUT_FILENAME", "庆祝国庆节快乐！.mp4")  # ← 要分析的文件名

if ACCOUNT_ID and (ACCOUNT_ID != ACCOUNT_ID.strip() or "/" in ACCOUNT_ID or "\\" in ACCOUNT_ID or ACCOUNT_ID in {".", ".."}):
    raise ValueError("账号 ID 不能包含路径分隔符或目录跳转字符。")

ACCOUNT_MEDIA_DIR = MEDIA_SOURCE_DIR / ACCOUNT_ID if ACCOUNT_ID else MEDIA_SOURCE_DIR
INPUT_MEDIA = ACCOUNT_MEDIA_DIR / INPUT_FILENAME
MASTER_RESULTS_ROOT = PROJECT_ROOT / "results" / "master"

# ══════════════════════════════════════════════════════════════
#  ④ 运行前检查：列出所选文件夹中的媒体文件，并确认目标文件存在
# ══════════════════════════════════════════════════════════════
MEDIA_SUFFIXES = {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v",
                  ".wav", ".mp3", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".aiff", ".aif"}

print(f"媒体根目录：{MEDIA_SOURCE_DIR}")
print(f"账号 ID：  {ACCOUNT_ID or '（未设置，使用媒体根目录）'}")
print(f"读取位置： {ACCOUNT_MEDIA_DIR}")

available = sorted(p.name for p in ACCOUNT_MEDIA_DIR.iterdir() if p.suffix.lower() in MEDIA_SUFFIXES) if ACCOUNT_MEDIA_DIR.is_dir() else []
if not available:
    raise FileNotFoundError(
        f"在 {ACCOUNT_MEDIA_DIR} 中没有找到可分析的媒体文件。\n"
        f"请将视频/音频放入该文件夹，或修改上面的 ACCOUNT_ID。"
    )

print(f"\n该文件夹中共有 {len(available)} 个媒体文件（最多显示前 20 个）：")
for name in available[:20]:
    mark = "  ← 当前选中" if name == INPUT_FILENAME else ""
    print(f"  {name}{mark}")
if len(available) > 20:
    print(f"  … 其余 {len(available) - 20} 个未显示")

if not INPUT_MEDIA.is_file():
    raise FileNotFoundError(
        f"找不到所选文件：{INPUT_FILENAME}\n"
        f"请把上面的 INPUT_FILENAME 改为列表中的某个文件名。"
    )

print(f"\n✔ 已选中：{INPUT_MEDIA}")
print(f"  结果将保存至：{MASTER_RESULTS_ROOT / ACCOUNT_ID / INPUT_MEDIA.stem if ACCOUNT_ID else MASTER_RESULTS_ROOT / INPUT_MEDIA.stem}")


# 本地媒体设置与 Demucs 音轨分离


In [ ]:
import re
import shutil
import subprocess
import soundfile as sf

if not INPUT_MEDIA.is_file():
    raise FileNotFoundError(f"请将 INPUT_MEDIA 设置为一个已存在的本地媒体文件：{INPUT_MEDIA}")
if shutil.which("ffmpeg") is None:
    raise FileNotFoundError("准备本地媒体音频需要安装 FFmpeg。")

MASTER_RESULTS_DIR = MASTER_RESULTS_ROOT / ACCOUNT_ID / INPUT_MEDIA.stem if ACCOUNT_ID else MASTER_RESULTS_ROOT / INPUT_MEDIA.stem
MASTER_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
video_suffixes = {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"}
audio_suffixes = {".wav", ".mp3", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".aiff", ".aif"}
if INPUT_MEDIA.suffix.lower() in video_suffixes:
    SOURCE_AUDIO = MASTER_RESULTS_DIR / "audio_original.wav"
    subprocess.run([
        "ffmpeg", "-y", "-i", str(INPUT_MEDIA), "-vn",
        "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(SOURCE_AUDIO),
    ], check=True)
elif INPUT_MEDIA.suffix.lower() in audio_suffixes:
    SOURCE_AUDIO = INPUT_MEDIA
else:
    raise ValueError(f"不支持的本地媒体文件扩展名：{INPUT_MEDIA.suffix}")

# 检测持续的尾部静音；保留完整源文件，并使用裁剪后的副本进行分析。
TRIM_TRAILING_SILENCE = True
SILENCE_THRESHOLD_DB = -40
MIN_TRAILING_SILENCE_SECONDS = 0.5
SILENCE_DETECT_MIN_SECONDS = 0.25
source_duration_s = float(sf.info(SOURCE_AUDIO).duration)
detection = subprocess.run([
    "ffmpeg", "-hide_banner", "-i", str(SOURCE_AUDIO),
    "-af", f"silencedetect=noise={SILENCE_THRESHOLD_DB}dB:d={SILENCE_DETECT_MIN_SECONDS}",
    "-f", "null", "-",
], check=True, capture_output=True, text=True)
events = re.findall(r"silence_(start|end):\s*([0-9.]+)", detection.stderr)
open_silence_start = None
trailing_silence_start = None
for event, time_value in events:
    event_time = float(time_value)
    if event == "start":
        open_silence_start = event_time
    elif open_silence_start is not None:
        if event_time >= source_duration_s - 0.1:
            trailing_silence_start = open_silence_start
        open_silence_start = None
if open_silence_start is not None and open_silence_start < source_duration_s:
    trailing_silence_start = open_silence_start

INPUT_AUDIO = SOURCE_AUDIO
if TRIM_TRAILING_SILENCE and trailing_silence_start is not None:
    silence_duration_s = source_duration_s - trailing_silence_start
    if silence_duration_s >= MIN_TRAILING_SILENCE_SECONDS:
        if trailing_silence_start <= 0:
            raise ValueError("静音检测发现源文件结束前没有可用音频。")
        INPUT_AUDIO = MASTER_RESULTS_DIR / "audio_analysis_trimmed.wav"
        subprocess.run([
            "ffmpeg", "-y", "-i", str(SOURCE_AUDIO), "-t", f"{trailing_silence_start:.6f}",
            "-vn", "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(INPUT_AUDIO),
        ], check=True)
        print(f"检测到 {silence_duration_s:.3f} 秒尾部静音；分析副本将在 {trailing_silence_start:.3f} 秒处结束。")
    else:
        print(f"尾部静音为 {silence_duration_s:.3f} 秒，短于设置的最短裁剪时长；保留完整音频。")
else:
    print("未检测到符合条件的尾部静音；保留完整音频。")

DEMUX_OUTPUT_ROOT = MASTER_RESULTS_DIR / "demucs"
DEMUX_STEM_DIR = DEMUX_OUTPUT_ROOT / "htdemucs" / INPUT_AUDIO.stem
files = {
    "original_mix": INPUT_AUDIO,
    "vocals": DEMUX_STEM_DIR / "vocals.wav",
    "no_vocals": DEMUX_STEM_DIR / "no_vocals.wav",
}
output_dir = MASTER_RESULTS_DIR
print(f"本地源文件：{INPUT_MEDIA}")
print(f"保留的源音频：{SOURCE_AUDIO}")
print(f"用于分析的音频：{INPUT_AUDIO}")
print(f"结果目录：{MASTER_RESULTS_DIR}")

In [ ]:
# 将配置的输入分离为人声与伴奏音轨。
import subprocess
import sys

if not INPUT_AUDIO.is_file():
    raise FileNotFoundError(f"找不到输入音频：{INPUT_AUDIO}")
DEMUX_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, "-m", "demucs",
    "-n", "htdemucs",
    "--two-stems=vocals",
    "-o", str(DEMUX_OUTPUT_ROOT),
    str(INPUT_AUDIO),
], check=True)

missing = [str(path) for path in files.values() if not path.is_file()]
if missing:
    raise FileNotFoundError("Demucs 未生成预期的音频文件：" + ", ".join(missing))
print("已生成以下音频版本：")
for part, path in files.items():
    print(f"{part}：{path}")

## 整轨摘要

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import librosa

files = {
    "original_mix": INPUT_AUDIO,
    "vocals": DEMUX_STEM_DIR / "vocals.wav",
    "no_vocals": DEMUX_STEM_DIR / "no_vocals.wav",
}

pitch_classes = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
summary, chroma_rows = [], []

for part, path in files.items():
    if not path.exists():
        print(f"跳过缺失文件：{path}")
        continue

    y, sr = librosa.load(path, sr=None, mono=True)
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    tempo = librosa.feature.tempo(y=y, sr=sr)[0]

    summary.append({
        "part": part,
        "file": str(path),
        "duration_s": round(len(y) / sr, 2),
        "sample_rate": sr,
        "rms_mean": float(librosa.feature.rms(y=y).mean()),
        "tempo_bpm_estimate": float(tempo),
        "spectral_contrast_mean": float(contrast.mean()),
        **{f"mfcc_{i+1}_mean": float(mfcc[i].mean()) for i in range(13)},
    })
    chroma_rows.extend(
        {"part": part, "pitch_class": note, "mean_chroma": float(chroma[i].mean())}
        for i, note in enumerate(pitch_classes)
    )

results = MASTER_RESULTS_DIR
results.mkdir(parents=True, exist_ok=True)
pd.DataFrame(summary).to_csv(results / "audio_feature_summary.csv", index=False)
pd.DataFrame(chroma_rows).to_csv(results / "audio_chroma_profile.csv", index=False)

pd.DataFrame(summary)

## 音频清单与来源记录

为每个输入音频版本创建一条可追溯记录。如适用，请填写来源网址和获取时间；源文件及各版本的校验和可作为稳定标识。

In [ ]:
import pandas as pd
import soundfile as sf

from datetime import datetime, timezone
from hashlib import sha256
from importlib.metadata import version

def file_sha256(path, chunk_size=1024 * 1024):
    digest = sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_media_sha256 = file_sha256(INPUT_MEDIA)
source_audio_sha256 = file_sha256(SOURCE_AUDIO)
analysis_audio_sha256 = file_sha256(INPUT_AUDIO)
sample_id = source_media_sha256[:16]
manifest_generated_at = datetime.now(timezone.utc).isoformat()
demucs_version = version("demucs")
manifest_rows = []

for stem_name, stem_path in files.items():
    info = sf.info(stem_path)
    is_original = stem_name == "original_mix"
    manifest_rows.append({
        "sample_id": sample_id,
        "account_id": ACCOUNT_ID,
        "title": INPUT_MEDIA.stem,
        "source_url": "",
        "search_query": "",
        "retrieved_at_utc": "",
        "manifest_generated_at_utc": manifest_generated_at,
        "source_path": str(INPUT_MEDIA.resolve()),
        "source_media_sha256": source_media_sha256,
        "analysis_audio_path": str(INPUT_AUDIO.resolve()),
        "source_audio_sha256": source_audio_sha256,
        "analysis_audio_sha256": analysis_audio_sha256,
        "audio_variant": stem_name,
        "audio_path": str(stem_path.resolve()),
        "parent_audio_path": "" if is_original else str(INPUT_AUDIO.resolve()),
        "separation_model": "" if is_original else "htdemucs",
        "separation_target": "" if is_original else "vocals",
        "separation_parameters": "" if is_original else "--two-stems=vocals",
        "demucs_version": demucs_version if not is_original else "",
        "duration_seconds": float(info.duration),
        "sample_rate_hz": int(info.samplerate),
        "channels": int(info.channels),
        "audio_file_size_bytes": int(stem_path.stat().st_size),
        "audio_sha256": file_sha256(stem_path),
        "feature_window_seconds": 5.0,
        "feature_hop_seconds": 2.5,
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = MASTER_RESULTS_DIR / "audio_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
manifest_files = sorted(MASTER_RESULTS_ROOT.rglob("audio_manifest.csv"))
manifest_tables = [pd.read_csv(path, dtype={"account_id": str, "sample_id": str}) for path in manifest_files]
for table in manifest_tables:
    if "account_id" not in table.columns:
        table.insert(1, "account_id", "")
all_manifests = pd.concat(manifest_tables, ignore_index=True, sort=False)
all_manifest_path = MASTER_RESULTS_ROOT / "audio_manifest_all.csv"
all_manifests.to_csv(all_manifest_path, index=False)
print(f"已将 {len(manifest_df)} 条音频记录保存至 {manifest_path}")
print(f"项目级汇总清单已更新：{all_manifest_path}")
print(manifest_df[["sample_id", "account_id", "audio_variant", "audio_path", "duration_seconds", "sample_rate_hz", "channels"]].to_string(index=False))

## 音频质量检查

检查每个音频版本的文件元数据、削波、近静音、均方根（RMS）和波形。这些是技术诊断指标，不是内容标签。

In [ ]:
import json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt

quality_dir = MASTER_RESULTS_DIR / "quality"
quality_dir.mkdir(parents=True, exist_ok=True)
audio_variants = {}
quality_rows = []
silence_threshold = 1e-4

for stem_name, stem_path in files.items():
    audio_info = sf.info(stem_path)
    y, sr = librosa.load(stem_path, sr=None, mono=True)
    duration_s = len(y) / sr
    peak = float(np.max(np.abs(y))) if len(y) else 0.0
    frame_rms = librosa.feature.rms(y=y)[0]
    silence_fraction = float(np.mean(frame_rms < silence_threshold)) if len(frame_rms) else 0.0
    quality = {
        "stem": stem_name,
        "file": str(stem_path.resolve()),
        "duration_seconds": duration_s,
        "sample_rate_hz": int(audio_info.samplerate),
        "channels": int(audio_info.channels),
        "frames": int(audio_info.frames),
        "peak_absolute_amplitude": peak,
        "clipped_sample_fraction": float(np.mean(np.abs(y) >= 0.999)) if len(y) else 0.0,
        "rms_mean": float(np.sqrt(np.mean(y ** 2))) if len(y) else 0.0,
        "near_silence_frame_fraction": silence_fraction,
        "near_silence_rms_threshold": silence_threshold,
    }
    (quality_dir / f"audio_quality_{stem_name}.json").write_text(
        json.dumps(quality, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    time = np.arange(len(y)) / sr
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(time, y, linewidth=0.35)
    ax.set(xlabel="时间（秒）", ylabel="振幅", title=f"波形：{stem_name}")
    fig.tight_layout()
    fig.savefig(quality_dir / f"audio_waveform_{stem_name}.png", dpi=150)
    plt.show()
    audio_variants[stem_name] = {"path": stem_path, "y": y, "sr": sr, "duration_s": duration_s}
    quality_rows.append(quality)

quality_df = pd.DataFrame(quality_rows)
display(quality_df)
quality_csv = quality_dir / "audio_quality_summary.csv"
quality_df.to_csv(quality_csv, index=False)
print("质量摘要已保存至：", quality_csv)


## 分窗手工特征

以 2.5 秒步长，从每个 5 秒窗口提取相同的 62 项可解释特征。每行窗口数据均保留音频版本和时间戳，便于后续比较或聚类。

In [ ]:
window_seconds = 5.0
hop_seconds = 2.5
min_tail_seconds = 0.5
all_feature_frames = []

def summarize_feature(name, values, row):
    values = np.asarray(values, dtype=float)
    row[f"{name}_mean"] = float(np.mean(values))
    row[f"{name}_std"] = float(np.std(values))

for stem_name, variant in audio_variants.items():
    y = variant["y"]
    sr = variant["sr"]
    duration_s = variant["duration_s"]
    feature_rows = []
    for start_s in np.arange(0.0, duration_s, hop_seconds):
        end_s = min(start_s + window_seconds, duration_s)
        if end_s - start_s < min_tail_seconds:
            continue
        start_sample = int(round(start_s * sr))
        end_sample = min(int(round(end_s * sr)), len(y))
        segment = y[start_sample:end_sample]
        if len(segment) == 0:
            continue

        row = {"stem": stem_name, "start_seconds": float(start_s), "end_seconds": float(end_s)}
        mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
        for i in range(mfcc.shape[0]):
            row[f"mfcc_{i + 1}_mean"] = float(np.mean(mfcc[i]))
            row[f"mfcc_{i + 1}_std"] = float(np.std(mfcc[i]))

        chroma = librosa.feature.chroma_stft(y=segment, sr=sr)
        for i in range(chroma.shape[0]):
            row[f"chroma_{i}"] = float(np.mean(chroma[i]))

        summarize_feature("rms", librosa.feature.rms(y=segment)[0], row)
        summarize_feature("spectral_centroid", librosa.feature.spectral_centroid(y=segment, sr=sr)[0], row)
        summarize_feature("spectral_bandwidth", librosa.feature.spectral_bandwidth(y=segment, sr=sr)[0], row)
        summarize_feature("spectral_rolloff", librosa.feature.spectral_rolloff(y=segment, sr=sr)[0], row)
        summarize_feature("zero_crossing_rate", librosa.feature.zero_crossing_rate(y=segment)[0], row)
        contrast = librosa.feature.spectral_contrast(y=segment, sr=sr)
        for i in range(contrast.shape[0]):
            row[f"spectral_contrast_{i}_mean"] = float(np.mean(contrast[i]))
            row[f"spectral_contrast_{i}_std"] = float(np.std(contrast[i]))
        feature_rows.append(row)

    stem_df = pd.DataFrame(feature_rows)
    feature_count = len([name for name in stem_df.columns if name not in {"stem", "start_seconds", "end_seconds"}])
    if feature_count != 62:
        raise ValueError(f"音轨 {stem_name} 应包含 62 项数值特征，实际为 {feature_count} 项")
    stem_feature_path = MASTER_RESULTS_DIR / f"handcrafted_audio_vectors_{stem_name}.csv"
    stem_df.to_csv(stem_feature_path, index=False)
    all_feature_frames.append(stem_df)
    print(f"{stem_name}：{len(stem_df)} 个窗口 × {stem_df.shape[1] - 3} 项特征 -> {stem_feature_path}")

feature_df = pd.concat(all_feature_frames, ignore_index=True)
combined_feature_path = MASTER_RESULTS_DIR / "handcrafted_audio_vectors_all_stems.csv"
feature_df.to_csv(combined_feature_path, index=False)
print("合并后的行数：", len(feature_df))
print("特征维度（不含音轨和时间戳）：", feature_df.shape[1] - 3)
print("合并表已保存至：", combined_feature_path)
print(feature_df.groupby("stem").size())
print(feature_df.head().to_string(index=False))

## 多音频版本音乐分析

对每个已配置的音频版本进行节拍跟踪、色度/调性候选分析、音级时间线、粗略结构分析、重复模式图，以及原始混音与分离音轨之间的色度 DTW 比较。这些估计结果需要结合聆听和人工检查。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import display

music_out_dir = MASTER_RESULTS_DIR / "music_analysis"
music_out_dir.mkdir(parents=True, exist_ok=True)
analysis_sr = 22050
chroma_hop = 2048
pitch_classes = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
major_template = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
minor_template = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
chroma_by_stem = {}
music_summary_rows = []

for stem_name, variant in audio_variants.items():
    y, sr = librosa.load(variant["path"], sr=analysis_sr, mono=True)
    stem_dir = music_out_dir / stem_name
    stem_dir.mkdir(parents=True, exist_ok=True)
    duration_s = len(y) / sr

    # Beat tracking, plus intervals between detected beats as a local tempo cue.
    beat_hop = 512
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=beat_hop)
    tempo, beat_frames = librosa.beat.beat_track(
        onset_envelope=onset_env, sr=sr, hop_length=beat_hop, units="frames"
    )
    beat_times = librosa.frames_to_time(beat_frames, sr=sr, hop_length=beat_hop)
    tempo_bpm = float(np.asarray(tempo).reshape(-1)[0])
    beat_intervals = np.diff(beat_times)
    local_bpm = np.divide(
        60.0, beat_intervals, out=np.full(beat_intervals.shape, np.nan), where=beat_intervals > 0
    )
    beat_df = pd.DataFrame({
        "beat_number": np.arange(1, len(beat_times) + 1),
        "time_seconds": beat_times,
        "interval_from_previous_seconds": np.r_[np.nan, beat_intervals],
        "local_bpm": np.r_[np.nan, local_bpm],
    })
    beat_df.to_csv(stem_dir / "beat_times.csv", index=False)
    onset_times = librosa.frames_to_time(np.arange(len(onset_env)), sr=sr, hop_length=beat_hop)
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(onset_times, onset_env, linewidth=0.7, label="起音强度")
    if len(beat_times):
        ax.scatter(beat_times, np.interp(beat_times, onset_times, onset_env), s=14, color="tab:red", label="检测到的节拍")
    ax.set(title=f"节拍跟踪：{stem_name}（速度估计 {tempo_bpm:.1f} BPM）", xlabel="时间（秒）", ylabel="起音强度")
    ax.legend()
    fig.tight_layout()
    fig.savefig(stem_dir / "beat_tracking.png", dpi=140)
    plt.show()

    # 色度分布、启发式调性候选，以及 5 秒粒度的音级时间线。
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=chroma_hop)
    chroma_by_stem[stem_name] = chroma
    chroma_profile = chroma.mean(axis=1)
    key_rows = []
    for mode, template in [("maj", major_template), ("min", minor_template)]:
        for tonic_index, tonic in enumerate(pitch_classes):
            correlation = np.corrcoef(chroma_profile, np.roll(template, tonic_index))[0, 1]
            key_rows.append({
                "key": f"{tonic}:{mode}", "tonic": tonic, "mode": mode,
                "profile_correlation": float(correlation),
            })
    key_df = pd.DataFrame(key_rows).sort_values("profile_correlation", ascending=False).reset_index(drop=True)
    key_df.to_csv(stem_dir / "key_candidates.csv", index=False)
    pd.DataFrame({"pitch_class": pitch_classes, "mean_chroma": chroma_profile}).to_csv(
        stem_dir / "chroma_profile.csv", index=False
    )
    frame_times = librosa.frames_to_time(np.arange(chroma.shape[1]), sr=sr, hop_length=chroma_hop)
    window_ids = np.floor(frame_times / 5.0).astype(int)
    timeline_rows = []
    for window_id in np.unique(window_ids):
        in_window = window_ids == window_id
        window_chroma = chroma[:, in_window]
        dominant_index = int(window_chroma.mean(axis=1).argmax())
        timeline_rows.append({
            "start_seconds": float(window_id * 5.0),
            "end_seconds": float(min((window_id + 1) * 5.0, duration_s)),
            "dominant_pitch_class": pitch_classes[dominant_index],
            "mean_chroma": float(window_chroma[dominant_index].mean()),
        })
    pd.DataFrame(timeline_rows).to_csv(stem_dir / "pitch_class_timeline_5s.csv", index=False)

    # 基于色度的粗略边界，并生成包含音频结束位置的完整区间表。
    n_frames = chroma.shape[1]
    boundary_frames = librosa.segment.agglomerative(chroma, k=min(8, n_frames))
    candidate_times = librosa.frames_to_time(boundary_frames, sr=sr, hop_length=chroma_hop)
    pd.DataFrame({"boundary_frame": boundary_frames, "time_seconds": candidate_times}).to_csv(
        stem_dir / "candidate_segment_boundaries.csv", index=False
    )
    interval_frames = np.unique(np.r_[0, boundary_frames, n_frames])
    interval_times = librosa.frames_to_time(interval_frames, sr=sr, hop_length=chroma_hop)
    interval_times[0] = 0.0
    interval_times[-1] = duration_s
    pd.DataFrame({
        "segment_id": np.arange(1, len(interval_times)),
        "start_seconds": interval_times[:-1],
        "end_seconds": interval_times[1:],
    }).to_csv(stem_dir / "candidate_segment_intervals.csv", index=False)
    recurrence = librosa.segment.recurrence_matrix(chroma, mode="affinity", self=True, width=3)
    fig, ax = plt.subplots(figsize=(6, 5))
    librosa.display.specshow(recurrence, x_axis="time", y_axis="time", sr=sr, hop_length=chroma_hop, cmap="magma", ax=ax)
    ax.set_title(f"色度自相似性：{stem_name}")
    fig.tight_layout()
    fig.savefig(stem_dir / "recurrence_matrix.png", dpi=140)
    plt.show()

    # 保存简洁的色度图以供目视检查。
    fig, ax = plt.subplots(figsize=(14, 4))
    img = librosa.display.specshow(chroma, x_axis="time", y_axis="chroma", sr=sr, hop_length=chroma_hop, ax=ax)
    ax.set_title(f"CQT 色度：{stem_name}")
    fig.colorbar(img, ax=ax)
    fig.tight_layout()
    fig.savefig(stem_dir / "chroma_timeline.png", dpi=140)
    plt.show()

    best = key_df.iloc[0]
    margin = float(best["profile_correlation"] - key_df.iloc[1]["profile_correlation"])
    music_summary_rows.append({
        "stem": stem_name, "duration_seconds": duration_s, "estimated_tempo_bpm": tempo_bpm,
        "detected_beats": len(beat_times), "top_key_candidate": best["key"],
        "top_key_correlation": float(best["profile_correlation"]), "top_key_margin": margin,
        "candidate_boundaries": len(boundary_frames),
    })

music_summary = pd.DataFrame(music_summary_rows)
music_summary.to_csv(music_out_dir / "music_summary.csv", index=False)
display(music_summary)

# 使用归一化色度 DTW 比较原始混音与各 Demucs 分离音轨。
alignment_rows = []
reference_name = "original_mix"
for candidate_name in ["vocals", "no_vocals"]:
    X = librosa.util.normalize(chroma_by_stem[reference_name], axis=0)
    Y = librosa.util.normalize(chroma_by_stem[candidate_name], axis=0)
    D, wp = librosa.sequence.dtw(X=X, Y=Y, metric="cosine")
    alignment_cost = float(D[-1, -1] / max(len(wp), 1))
    alignment_df = pd.DataFrame({
        "original_frame": wp[::-1, 0], "candidate_frame": wp[::-1, 1],
        "original_time_seconds": librosa.frames_to_time(wp[::-1, 0], sr=analysis_sr, hop_length=chroma_hop),
        "candidate_time_seconds": librosa.frames_to_time(wp[::-1, 1], sr=analysis_sr, hop_length=chroma_hop),
    })
    alignment_df.to_csv(music_out_dir / f"dtw_original_vs_{candidate_name}.csv", index=False)
    alignment_rows.append({"reference": reference_name, "candidate": candidate_name, "normalized_dtw_cost": alignment_cost})
pd.DataFrame(alignment_rows).to_csv(music_out_dir / "dtw_summary.csv", index=False)
display(pd.DataFrame(alignment_rows))
print("音乐分析结果已保存至：", music_out_dir)

## 分窗音乐向量：手工特征基线与可选的 MERT

上面的摘要特征大多是整轨统计量。要比较不同音乐片段，需要为每个带时间戳的窗口生成一个固定长度的向量。测试视频笔记本以原始混音、人声和伴奏音轨为例，使用 5 秒窗口和 2.5 秒步长。其 62 项手工特征（MFCC、色度、能量、频谱统计量）是可解释的基线，并非经过验证的最优特征集。

如需使用学习得到的音乐表示，可运行下方可选单元：它使用 `m-a-p/MERT-v1-95M` 对每个可用音轨窗口进行编码（5 秒上下文，24 kHz 输入），并对最后一层隐藏状态取均值。请先在当前笔记本内核中安装兼容版本的 `transformers` 和 PyTorch。运行 `from_pretrained` 会下载模型代码和权重；`trust_remote_code=True` 会执行仓库中的代码，因此运行前请核查其来源、许可证和代码。此步骤为可选项，只有明确运行该单元时才会执行。

输出包括一个向量矩阵，以及记录每个向量源路径和时间范围的索引表。MERT 嵌入、手工特征、调性/速度摘要和 Basic Pitch 音符事件属于不同表示方式；考虑融合前应先分别比较。处理大型语料时，请对所有文件使用相同的模型、采样率、分窗方式和池化方法，并通过聆听检查最近邻结果。这些向量反映的是模型或特征空间中的相似性，并不代表音乐标签，也不构成对宣传内容的判断。

In [ ]:
# 可选：在“整轨摘要”单元定义 `files` 映射后运行。
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import torch
from transformers import AutoModel, Wav2Vec2FeatureExtractor

MERT_MODEL_ID = "m-a-p/MERT-v1-95M"
mert_processor = Wav2Vec2FeatureExtractor.from_pretrained(
    MERT_MODEL_ID, trust_remote_code=True
)
mert_model = AutoModel.from_pretrained(
    MERT_MODEL_ID, trust_remote_code=True
)
mert_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mert_model = mert_model.to(mert_device).eval()
mert_sr = int(mert_processor.sampling_rate)
window_seconds, hop_seconds = 5.0, 2.5
min_tail_seconds = 0.5
embedding_rows, embedding_vectors = [], []

for part, path in files.items():
    if not path.is_file():
        raise FileNotFoundError(f"找不到音频版本 {part!r} 对应的文件：{path}")
    audio, source_sr = librosa.load(path, sr=None, mono=True)
    duration_s = len(audio) / source_sr
    audio = librosa.resample(audio, orig_sr=source_sr, target_sr=mert_sr).astype(np.float32)
    window_samples = int(round(window_seconds * mert_sr))

    for start_s in np.arange(0.0, duration_s, hop_seconds):
        end_s = min(start_s + window_seconds, duration_s)
        actual_duration_s = end_s - start_s
        if actual_duration_s < window_seconds:
            continue
        start_sample = int(round(start_s * mert_sr))
        end_sample = min(int(round(end_s * mert_sr)), len(audio))
        segment = audio[start_sample:end_sample]
        padded_samples = 0

        inputs = mert_processor(segment, sampling_rate=mert_sr, return_tensors="pt")
        inputs = {name: value.to(mert_device) for name, value in inputs.items()}
        with torch.inference_mode():
            outputs = mert_model(**inputs)
            hidden_state = getattr(outputs, "last_hidden_state", None)
            if hidden_state is None:
                hidden_states = getattr(outputs, "hidden_states", None)
                if not hidden_states:
                    raise RuntimeError("MERT 未返回 last_hidden_state 或 hidden_states。")
                hidden_state = hidden_states[-1]
            vector = hidden_state.mean(dim=1).squeeze(0).cpu().numpy().astype(np.float32)

        embedding_vectors.append(vector)
        embedding_rows.append({
            "embedding_row": len(embedding_rows),
            "sample_id": sample_id,
            "account_id": ACCOUNT_ID,
            "title": INPUT_MEDIA.stem,
            "part": part,
            "source_path": str(path),
            "start_seconds": float(start_s),
            "end_seconds": float(end_s),
            "actual_duration_seconds": float(actual_duration_s),
            "padded_duration_seconds": float(padded_samples / mert_sr),
            "model_id": MERT_MODEL_ID,
            "sample_rate_hz": mert_sr,
            "pooling": "last_hidden_state_mean_over_time",
        })

if not embedding_vectors:
    raise RuntimeError("未能编码任何音频窗口；请检查输入文件和音频时长。")

mert_matrix = np.stack(embedding_vectors)
mert_output_dir = MASTER_RESULTS_DIR / "mert"
mert_output_dir.mkdir(parents=True, exist_ok=True)
np.save(mert_output_dir / "mert_v1_95m_embeddings.npy", mert_matrix)
mert_index = pd.DataFrame(embedding_rows)
mert_index.to_csv(mert_output_dir / "mert_v1_95m_embedding_index.csv", index=False)
print("运行设备：", mert_device)
print("嵌入矩阵形状：", mert_matrix.shape)
print("输出已保存至：", mert_output_dir)

## MERT 向量写入 Milvus 向量库（可选）

将当前视频的平均 MERT 向量增量写入 `results/milvus_mert.db`。
同一视频重复运行会覆盖旧向量，未变化的视频在批量增量导入时自动跳过。
未安装 `pymilvus` 时自动跳过，不影响主流程。

In [ ]:
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
try:
    from milvus_store import upsert_video
    upsert_video(MASTER_RESULTS_DIR)
except ImportError:
    print("未安装 pymilvus，跳过向量入库")


## 音频窗口的探索性聚类

请在手工特征和 MERT 单元运行后执行。此步骤分别对两种表示进行聚类，测试 k=2–6 的 K-means，并保存 k=3 的标签和 PCA 图。这些有重叠的窗口来自同一视频及其三个相关音轨，因此聚类结果仅表示探索性的声学相似分组，不是语义类别，也不是相互独立的观测样本。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler, normalize

cluster_dir = MASTER_RESULTS_DIR / "clustering"
cluster_dir.mkdir(parents=True, exist_ok=True)
metadata_columns = ["stem", "start_seconds", "end_seconds"]
feature_columns = [column for column in feature_df.columns if column not in metadata_columns]
if feature_df.empty or not feature_columns:
    raise ValueError("手工特征数据为空；请先运行分窗特征单元。")
if mert_matrix.shape[0] != len(mert_index):
    raise ValueError("MERT 矩阵的行数与索引表不一致。")

mert_metadata = mert_index.sort_values("embedding_row").reset_index(drop=True).rename(columns={"part": "stem"})
keys = ["stem", "start_seconds", "end_seconds"]
feature_metadata = feature_df[keys].copy()
index_metadata = mert_metadata[keys].copy()
feature_metadata["start_seconds"] = feature_metadata["start_seconds"].round(6)
feature_metadata["end_seconds"] = feature_metadata["end_seconds"].round(6)
index_metadata["start_seconds"] = index_metadata["start_seconds"].round(6)
index_metadata["end_seconds"] = index_metadata["end_seconds"].round(6)
aligned_feature_df = mert_metadata[keys].merge(
    feature_df, on=keys, how="left", validate="one_to_one", indicator=True
)
if aligned_feature_df["_merge"].ne("both").any():
    raise ValueError("某个 MERT 窗口找不到对应的手工特征行。")
aligned_feature_df = aligned_feature_df.drop(columns="_merge")

representations = {
    "handcrafted_62d": StandardScaler().fit_transform(aligned_feature_df[feature_columns].to_numpy(dtype=float)),
    "mert_768d": normalize(mert_matrix.astype(float), norm="l2"),
}
assignment_frames = []
metric_rows = []
composition_frames = []
markers = {"original_mix": "o", "vocals": "^", "no_vocals": "s"}

for representation_name, matrix in representations.items():
    if not np.isfinite(matrix).all():
        raise ValueError(f"表示 {representation_name} 的向量中存在非有限值。")
    max_k = min(6, len(matrix) - 1)
    for k in range(2, max_k + 1):
        labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(matrix)
        score = float(silhouette_score(matrix, labels))
        metric_rows.append({"representation": representation_name, "k": k, "silhouette": score})
    labels = KMeans(n_clusters=3, random_state=42, n_init=20).fit_predict(matrix)
    coordinates = PCA(n_components=2, random_state=42).fit_transform(matrix)
    if representation_name == "handcrafted_62d":
        metadata = aligned_feature_df[metadata_columns].reset_index(drop=True)
    else:
        metadata = mert_metadata[metadata_columns].reset_index(drop=True)
    assigned = metadata.copy()
    assigned["representation"] = representation_name
    assigned["cluster_k3"] = labels
    assigned["pca_1"] = coordinates[:, 0]
    assigned["pca_2"] = coordinates[:, 1]
    assignment_frames.append(assigned)
    composition = pd.crosstab(assigned["cluster_k3"], assigned["stem"], normalize="index")
    composition = composition.reset_index().melt(id_vars="cluster_k3", var_name="stem", value_name="fraction")
    composition["representation"] = representation_name
    composition_frames.append(composition)

    fig, ax = plt.subplots(figsize=(8, 6))
    for cluster_id in sorted(np.unique(labels)):
        for stem_name in assigned["stem"].unique():
            selected = (assigned["cluster_k3"] == cluster_id) & (assigned["stem"] == stem_name)
            if selected.any():
                ax.scatter(assigned.loc[selected, "pca_1"], assigned.loc[selected, "pca_2"],
                           marker=markers.get(stem_name, "o"), label=f"cluster {cluster_id} / {stem_name}", alpha=0.8)
    ax.set(title=f"{representation_name}：K-means k=3（PCA 投影）", xlabel="PCA 1", ylabel="PCA 2")
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(cluster_dir / f"{representation_name}_k3_pca.png", dpi=160)
    plt.close(fig)

assignments = pd.concat(assignment_frames, ignore_index=True)
metrics = pd.DataFrame(metric_rows)
composition = pd.concat(composition_frames, ignore_index=True)
assignments.to_csv(cluster_dir / "window_cluster_assignments.csv", index=False)
metrics.to_csv(cluster_dir / "kmeans_silhouette_sweep.csv", index=False)
composition.to_csv(cluster_dir / "k3_cluster_stem_composition.csv", index=False)
print("轮廓系数扫描（数值越高，表示该特征空间中的聚类越紧密、分离度越高；不代表语义评分）：")
print(metrics.pivot(index="k", columns="representation", values="silhouette").round(3).to_string())
print()
print("k=3 时的聚类规模与音轨构成：")
print(composition.pivot_table(index=["representation", "cluster_k3"], columns="stem", values="fraction", fill_value=0).round(2).to_string())
print("探索性聚类结果已保存至：", cluster_dir)


### 单音轨后续分析：仅使用伴奏音轨

此步骤仅提取 `no_vocals` 的 MERT 窗口，避免聚类结果受到不同 Demucs 音轨差异的影响。目前只有 12 个重叠窗口，因此结果仅适合探索同一音轨内部的变化，不能作为可靠的语义分类。

In [ ]:
# 仅对 no_vocals 的 MERT 窗口重新聚类，作为音轨内部的探索性检查。
no_vocals_mask = mert_index["part"].eq("no_vocals").to_numpy()
no_vocals_metadata = mert_index.loc[no_vocals_mask].sort_values("embedding_row").reset_index(drop=True).copy()
no_vocals_matrix = mert_matrix[no_vocals_mask].astype(float)
if len(no_vocals_matrix) < 3:
    raise ValueError(f"至少需要 3 个 no_vocals 窗口；当前找到 {len(no_vocals_matrix)} 个。")
no_vocals_matrix = normalize(no_vocals_matrix, norm="l2")
no_vocals_sweep = []
for k in range(2, min(6, len(no_vocals_matrix) - 1) + 1):
    sweep_labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(no_vocals_matrix)
    no_vocals_sweep.append({"k": k, "silhouette": float(silhouette_score(no_vocals_matrix, sweep_labels))})

no_vocals_labels = KMeans(n_clusters=2, random_state=42, n_init=20).fit_predict(no_vocals_matrix)
no_vocals_metadata["cluster_k2"] = no_vocals_labels
no_vocals_out_dir = cluster_dir / "no_vocals_only"
no_vocals_out_dir.mkdir(parents=True, exist_ok=True)
no_vocals_assignments = no_vocals_metadata[[
    "embedding_row", "part", "start_seconds", "end_seconds", "actual_duration_seconds", "cluster_k2"
]]
no_vocals_assignments.to_csv(no_vocals_out_dir / "mert_no_vocals_k2_assignments.csv", index=False)
no_vocals_metrics = pd.DataFrame(no_vocals_sweep)
no_vocals_metrics.to_csv(no_vocals_out_dir / "mert_no_vocals_silhouette_sweep.csv", index=False)
no_vocals_summary = no_vocals_metadata.groupby("cluster_k2").agg(
    window_count=("embedding_row", "size"),
    start_min=("start_seconds", "min"),
    end_max=("end_seconds", "max"),
).reset_index()
no_vocals_summary.to_csv(no_vocals_out_dir / "mert_no_vocals_cluster_summary.csv", index=False)
print("仅伴奏音轨的 MERT 轮廓系数扫描：")
print(no_vocals_metrics.round(3).to_string(index=False))
print("\nk=2 时的窗口分配：")
print(no_vocals_assignments[["start_seconds", "end_seconds", "cluster_k2"]].to_string(index=False))
print("\n仅伴奏音轨的聚类结果已保存至：", no_vocals_out_dir)


## 跨视频聚类：伴奏音轨的 MERT 窗口

此项目级实验会收集 `results/` 下已有的 MERT 索引，排除临时静音清理产物，仅选择时长完整的 5 秒 `no_vocals` 窗口，并同时报告按窗口数量加权的聚类结果和各视频均衡后的比较结果。至少需要三个不同的视频样本。窗口之间存在重叠；聚类仍仅表示探索性的声学相似性，不是语义或内容标签。

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize

project_results_dir = PROJECT_ROOT / "results"
corpus_rows = []
corpus_vectors = []
used_samples = set()
for index_path in sorted(project_results_dir.rglob("mert_v1_95m_embedding_index.csv")):
    relative_index = index_path.relative_to(project_results_dir)
    relative_text = str(relative_index).lower()
    if "trailing_silence_cleanup" in relative_text:
        continue
    vector_path = index_path.with_name("mert_v1_95m_embeddings.npy")
    if not vector_path.is_file():
        raise FileNotFoundError(f"MERT 索引找不到对应的向量矩阵：{vector_path}")
    index = pd.read_csv(index_path, dtype={"account_id": str, "sample_id": str})
    vectors = np.load(vector_path)
    if len(vectors) != len(index):
        raise ValueError(f"MERT 行数不匹配：{index_path}")
    stem_column = "part" if "part" in index.columns else "stem"
    if stem_column not in index.columns:
        raise ValueError(f"MERT 索引中没有音轨列：{index_path}")
    if index["model_id"].nunique() != 1 or index["model_id"].iloc[0] != "m-a-p/MERT-v1-95M":
        raise ValueError(f"{index_path} 中的 MERT 模型与预期不符；只能比较使用相同模型生成的数据。")
    if "sample_id" in index.columns and index["sample_id"].nunique(dropna=True) == 1:
        sample_id_value = str(index["sample_id"].dropna().iloc[0])
        account_id_value = str(index["account_id"].dropna().iloc[0]) if "account_id" in index.columns and index["account_id"].notna().any() else ""
        sample_name = f"{account_id_value}/{sample_id_value}" if account_id_value else sample_id_value
    else:
        sample_id_value = ""
        account_id_value = ""
        sample_name = relative_index.parts[0]
        if sample_name == "master":
            if len(relative_index.parts) < 3:
                raise ValueError(f"无法从 {relative_index} 识别主样本。")
            sample_name = "/".join(relative_index.parts[:2])
    if sample_name in used_samples:
        raise ValueError(f"样本 {sample_name} 存在重复的 MERT 索引；请先解决重复问题。")
    used_samples.add(sample_name)
    mask = index[stem_column].eq("no_vocals").to_numpy()
    if "actual_duration_seconds" in index.columns:
        mask = mask & index["actual_duration_seconds"].ge(4.999).to_numpy()
    mask = mask & (index["end_seconds"] - index["start_seconds"]).ge(4.999).to_numpy()
    selected = index.loc[mask].copy().reset_index(drop=True)
    if selected.empty:
        continue
    row_ids = selected["embedding_row"].to_numpy(dtype=int)
    selected_vectors = vectors[row_ids].astype(np.float32)
    for row_number, row in selected.iterrows():
        corpus_rows.append({
            "sample": sample_name,
            "sample_id": sample_id_value,
            "account_id": account_id_value,
            "index_file": str(relative_index),
            "audio_variant": "no_vocals",
            "start_seconds": float(row["start_seconds"]),
            "end_seconds": float(row["end_seconds"]),
            "model_id": row["model_id"],
        })
    corpus_vectors.extend(selected_vectors)

if len(used_samples) < 3:
    raise ValueError(f"至少需要 3 个不同样本的 MERT 数据；当前找到：{sorted(used_samples)}")
corpus_metadata = pd.DataFrame(corpus_rows)
corpus_matrix = normalize(np.stack(corpus_vectors).astype(float), norm="l2")
counts = corpus_metadata.groupby("sample").size().sort_index()
if counts.min() < 3:
    raise ValueError(f"每个样本至少需要 3 个完整窗口：{counts.to_dict()}")
corpus_out_dir = project_results_dir / "corpus_clustering" / "no_vocals_mert"
corpus_out_dir.mkdir(parents=True, exist_ok=True)

# 首先保留所有窗口；在此按时长加权的运行中，较长视频会贡献更多观测。
all_metric_rows = []
for k in range(2, min(6, len(corpus_matrix) - 1) + 1):
    labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(corpus_matrix)
    all_metric_rows.append({"run": "all_windows", "k": k,
                            "silhouette": float(silhouette_score(corpus_matrix, labels))})
all_labels = KMeans(n_clusters=3, random_state=42, n_init=20).fit_predict(corpus_matrix)
all_assignments = corpus_metadata.copy()
all_assignments["cluster_k3"] = all_labels
all_assignments.to_csv(corpus_out_dir / "all_windows_assignments.csv", index=False)

# 然后从每个视频中等间隔抽取相同数量的窗口。
balance_n = int(counts.min())
balanced_positions = []
for sample_name in sorted(counts.index):
    sample_positions = np.flatnonzero(corpus_metadata["sample"].to_numpy() == sample_name)
    chosen = np.linspace(0, len(sample_positions) - 1, balance_n).round().astype(int)
    balanced_positions.extend(sample_positions[chosen].tolist())
balanced_positions = np.asarray(balanced_positions, dtype=int)
balanced_matrix = corpus_matrix[balanced_positions]
balanced_metadata = corpus_metadata.iloc[balanced_positions].reset_index(drop=True)
balanced_metric_rows = []
for k in range(2, min(6, len(balanced_matrix) - 1) + 1):
    labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(balanced_matrix)
    balanced_metric_rows.append({"run": "balanced_per_video", "k": k,
                                 "silhouette": float(silhouette_score(balanced_matrix, labels))})
balanced_labels = KMeans(n_clusters=3, random_state=42, n_init=20).fit_predict(balanced_matrix)
balanced_assignments = balanced_metadata.copy()
balanced_assignments["cluster_k3"] = balanced_labels
balanced_assignments.to_csv(corpus_out_dir / "balanced_assignments.csv", index=False)

metrics = pd.DataFrame(all_metric_rows + balanced_metric_rows)
metrics.to_csv(corpus_out_dir / "silhouette_sweeps.csv", index=False)
composition = pd.concat([
    all_assignments.assign(run="all_windows"),
    balanced_assignments.assign(run="balanced_per_video"),
]).groupby(["run", "cluster_k3", "sample"]).size().rename("window_count").reset_index()
composition.to_csv(corpus_out_dir / "k3_sample_composition.csv", index=False)

# PCA 仅用于可视化；聚类使用的是原始归一化 MERT 向量。
plot_meta = all_assignments.copy()
plot_meta[["pca_1", "pca_2"]] = PCA(n_components=2, random_state=42).fit_transform(corpus_matrix)
fig, ax = plt.subplots(figsize=(9, 7))
for sample_name in sorted(plot_meta["sample"].unique()):
    selected = plot_meta["sample"].eq(sample_name)
    ax.scatter(plot_meta.loc[selected, "pca_1"], plot_meta.loc[selected, "pca_2"],
               c=plot_meta.loc[selected, "cluster_k3"], cmap="tab10", marker="o", alpha=0.75,
               label=sample_name)
ax.set(title="伴奏音轨 MERT 窗口：K-means k=3（PCA 投影）", xlabel="PCA 1", ylabel="PCA 2")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(corpus_out_dir / "all_windows_k3_pca.png", dpi=160)
plt.close(fig)

print("不同视频及符合条件的完整窗口数量：")
print(counts.to_string())
print(f"\n全窗口聚类使用 {len(corpus_matrix)} 个窗口；均衡比较每个视频使用 {balance_n} 个窗口（共 {len(balanced_matrix)} 个）。")
print("\n轮廓系数：")
print(metrics.pivot(index="k", columns="run", values="silhouette").round(3).to_string())
print("\nk=3 时的样本构成：")
print(composition.pivot_table(index=["run", "cluster_k3"], columns="sample", values="window_count", fill_value=0).to_string())
print("跨视频伴奏音轨聚类结果已保存至：", corpus_out_dir)


## 跨视频最近邻检索

请在跨视频聚类单元之后运行。针对每个完整的 5 秒 `no_vocals` 窗口，从其他每个视频中检索最多两个余弦相似窗口。同一候选视频中的匹配窗口在时间上不会重叠，以减少 2.5 秒窗口步长带来的重复结果。输出是带时间戳、供聆听和复核使用的候选列表，不是语义标签或置信度评分。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

top_matches_per_video = 2
similarity_matrix = corpus_matrix @ corpus_matrix.T
sample_values = corpus_metadata["sample"].to_numpy()
window_starts = corpus_metadata["start_seconds"].to_numpy(dtype=float)
window_ends = corpus_metadata["end_seconds"].to_numpy(dtype=float)
neighbor_rows = []

for query_position, query in corpus_metadata.iterrows():
    query_sample = query["sample"]
    other_samples = sorted(set(sample_values) - {query_sample})
    for candidate_sample in other_samples:
        candidate_positions = np.flatnonzero(sample_values == candidate_sample)
        ranked_positions = candidate_positions[
            np.argsort(similarity_matrix[query_position, candidate_positions])[::-1]
        ]
        selected_intervals = []
        for candidate_position in ranked_positions:
            candidate_start = window_starts[candidate_position]
            candidate_end = window_ends[candidate_position]
            overlaps_selected = any(
                max(candidate_start, selected_start) < min(candidate_end, selected_end)
                for selected_start, selected_end in selected_intervals
            )
            if overlaps_selected:
                continue
            selected_intervals.append((candidate_start, candidate_end))
            neighbor_rows.append({
                "query_sample": query_sample,
                "query_start_seconds": float(query["start_seconds"]),
                "query_end_seconds": float(query["end_seconds"]),
                "match_sample": candidate_sample,
                "match_start_seconds": float(candidate_start),
                "match_end_seconds": float(candidate_end),
                "rank_within_match_sample": len(selected_intervals),
                "cosine_similarity": float(similarity_matrix[query_position, candidate_position]),
            })
            if len(selected_intervals) == top_matches_per_video:
                break

nearest_neighbors = pd.DataFrame(neighbor_rows).sort_values(
    ["query_sample", "query_start_seconds", "cosine_similarity"],
    ascending=[True, True, False],
).reset_index(drop=True)
nearest_neighbors_path = corpus_out_dir / "cross_video_nearest_neighbors.csv"
nearest_neighbors.to_csv(nearest_neighbors_path, index=False)
print(f"已将 {len(nearest_neighbors)} 条带时间戳的跨视频匹配结果保存至 {nearest_neighbors_path}")
display(nearest_neighbors.head(12))

## 跨账号 BGM 复用检测

请在 Milvus 入库（上方可选单元）完成后运行。此项目级实验从 Milvus 向量库读取全部视频级 MERT 平均向量，计算两两余弦相似度，筛选**跨账号**的高相似视频对（默认阈值 0.98），用于发现不同账号复用同一 BGM 的情况。

- 库中向量已 L2 归一化，点积即余弦相似度；同一视频对自身已排除。
- 实测分布（372 条向量）：跨账号相似度中位数约 0.93，p99 约 0.98；≥0.99 的 27 对基本是同一新闻素材的互相转发（相似度 0.9998+）。因此默认阈值取 0.98，可在代码中调整。
- 结果为表示层面的声学相似，不构成内容来源或意图的结论。
- 只读取向量库，不修改库中数据；结果保存至 `results/corpus_clustering/cross_account_bgm/`。

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from milvus_store import COLLECTION, get_client

SIM_THRESHOLD = 0.98  # 跨账号余弦相似度阈值（实测跨账号中位数≈0.93，需取高位才有区分度）

client = get_client()
rows = client.query(
    COLLECTION,
    filter="npy_mtime >= 0",
    output_fields=["account_id", "video_id", "vector"],
    limit=10000,
)
print(f"从 Milvus 读取 {len(rows)} 条视频级向量")

vectors = np.array([r["vector"] for r in rows], dtype=np.float32)
accounts = np.array([r["account_id"] for r in rows])
video_ids = np.array([r["video_id"] for r in rows])

similarity = vectors @ vectors.T  # 已 L2 归一化，点积即余弦相似度
cross_account = accounts[:, None] != accounts[None, :]
upper_i, upper_j = np.triu_indices(len(rows), k=1)  # 每对只取一次且排除自身

pairs = [
    {
        "account_a": accounts[i],
        "video_a": video_ids[i],
        "account_b": accounts[j],
        "video_b": video_ids[j],
        "cosine_similarity": round(float(similarity[i, j]), 4),
    }
    for i, j in zip(upper_i, upper_j)
    if cross_account[i, j] and similarity[i, j] >= SIM_THRESHOLD
]
reuse_df = pd.DataFrame(pairs).sort_values("cosine_similarity", ascending=False)

bgm_out_dir = PROJECT_ROOT / "results" / "corpus_clustering" / "cross_account_bgm"
bgm_out_dir.mkdir(parents=True, exist_ok=True)
reuse_path = bgm_out_dir / "cross_account_bgm_reuse.csv"
reuse_df.to_csv(reuse_path, index=False)

print(f"阈值 {SIM_THRESHOLD} 下跨账号高相似对: {len(reuse_df)}")
print("结果已保存至:", reuse_path)
reuse_df.head(20)

## Basic Pitch 音符事件与 MIDI（可选）

请在已安装 Basic Pitch 命令行工具的环境中运行。每个已配置且可用的音频版本都会保存到各自的输出文件夹。

In [ ]:
# 使用 Basic Pitch 转录所有已配置的音频版本。
from pathlib import Path
import subprocess

basic_pitch = Path.home() / "venvs/basic-pitch/bin/basic-pitch"
if not basic_pitch.is_file():
    raise FileNotFoundError(f"找不到 Basic Pitch 命令行工具：{basic_pitch}")

for part, audio_path in files.items():
    if not audio_path.is_file():
        raise FileNotFoundError(f"找不到音频版本 {part!r} 对应的文件：{audio_path}")
    part_output_dir = MASTER_RESULTS_DIR / "basic_pitch" / part
    part_output_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        str(basic_pitch), "--save-midi", "--save-note-events",
        str(part_output_dir), str(audio_path),
    ], check=True)
    print(f"{part}：Basic Pitch 输出已保存至 {part_output_dir}")

## 9. 归档：git-annex 同步到 R2

将本次运行新增/更新的 .npy 向量通过 git-annex 上传到 R2（pika-audio/git-annex-data），并提交 annex 指针。幂等，可反复运行。

In [ ]:
import subprocess

PROJECT_ROOT = '/home/ubuntu/music'

def _run(cmd):
    r = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    tail = (r.stdout + r.stderr).strip().splitlines()[-3:]
    print(f"{' '.join(cmd[:3])}... -> rc={r.returncode}")
    for line in tail:
        print('   ', line[:150])
    return r.returncode

# 1. 新增 npy 纳入 annex 管理（.gitattributes 规则: *.npy）
_run(['git', 'annex', 'add', 'results/master'])

# 2. 上传内容到 R2（只传缺的，幂等）
_run(['git', 'annex', 'copy', '--to', 'r2', "--include=*.npy"])

# 3. 提交 annex 指针 + 清单 CSV/JSON
_run(['git', 'add', 'results/master'])
if _run(['git', 'diff', '--cached', '--quiet']) != 0:
    _run(['git', 'commit', '-m', 'chore: annex 新增分析产物并同步至 R2'])
else:
    print('无新变更，跳过提交')